In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_tego_notebooka = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"


def wyslij_odpowiedz(task_id, final_answer):
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_tego_notebooka,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


# BSM L01 — Podstawowe pojęcia bezpieczeństwa, modelowanie bezpieczeństwa, triada AIC

## Kontekst techniczny
Pracujesz na uproszczonym systemie **BSM-Lab** (aplikacja mobilna + API + usługi zewnętrzne).
Wszystkie zadania dotyczą trzech atrybutów bezpieczeństwa:
- **A** (Availability),
- **I** (Integrity),
- **C** (Confidentiality).

## Zasady pracy
1. Każde zadanie wykonuj w Pythonie.
2. Nie zmieniaj danych wejściowych podanych w treści.
3. W każdym zadaniu utwórz:
   - tabelę szczegółową (lista/dict lub `DataFrame`),
   - co najmniej 1 wykres,
   - `final_answer` w dokładnie wymaganym formacie.
4. Zaokrąglaj wartości zmiennoprzecinkowe do 2 miejsc: `round(x, 2)`.
5. Dla JSON używaj `canonical_json(obj)`.

## Materiały pomocnicze
- OWASP Threat Modeling Cheat Sheet: https://cheatsheetseries.owasp.org/cheatsheets/Threat_Modeling_Cheat_Sheet.html
- NIST SP 800-12r1 (C/I/A): https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-12r1.pdf
- Android Security Best Practices: https://developer.android.com/privacy-and-security/security-best-practices


In [ ]:
import json
import hashlib


def canonical_json(obj):
    """Kanoniczny JSON: stały porządek kluczy i brak ASCII-escaping."""
    return json.dumps(obj, sort_keys=True, ensure_ascii=False)


def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


answers = {}


def zapisz_i_wyslij(task_id, final_answer):
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    wyslij_odpowiedz(task_id, final_answer)


# BSM01 — Poufność: wyciek danych w logach i jego ograniczenie

### Co robisz
Symulujesz trzy warianty logowania i mierzysz wyciek jawnych danych.

### Kroki
1. Użyj `LOG_EVENTS`.
2. Zaimplementuj 3 warianty:
   - `vulnerable_log(event)`:
     - `email_vis = pełny email`,
     - `token_vis = pełny token`.
   - `secure_log_v1(event)`:
     - `email_vis`: zostaw tylko pierwszy znak części lokalnej, resztę zamień na `*`, domenę zostaw bez zmian.
       Przykład: `jan.kowalski@example.com -> j***********@example.com`.
     - `token_vis`: zostaw pierwsze 4 i ostatnie 4 znaki, środek zamień na `...`.
       Przykład: `4f91ab...1d22`.
   - `secure_log_v2(event)`:
     - `email_vis` jak w v1,
     - `token_vis = "<redacted>"`.
3. Każdy wariant ma zwracać słownik z kluczami: `uid`, `action`, `email_vis`, `token_vis`.
4. Utwórz listy: `logs_vuln`, `logs_v1`, `logs_v2`.
5. Definicja wycieku (obowiązkowa):
   - liczysz jawne znaki **tylko** na polach `email_vis` i `token_vis`,
   - jeśli `token_vis == "<redacted>"`, to wkład tokenu do wycieku = 0 (cały literal, łącznie z `<` i `>`, liczy się jako zamaskowany),
   - w pozostałych przypadkach znak `*` i `.` nie liczą się jako jawne,
   - jawne są litery/cyfry oraz znaki specjalne poza powyższymi maskami.
6. Zbuduj tabelę i wykres sumarycznego wycieku dla `vuln/v1/v2`.

### Sprawdź poprawność
- `logs_vuln`, `logs_v1`, `logs_v2` mają po 6 wpisów.
- W `v2` wyciek tokenu musi wynosić 0.
- `leak_vuln > leak_v1 >= leak_v2`.

### Dane
```python
LOG_EVENTS = [
    {"uid": 1, "email": "jan.kowalski@example.com", "token": "4f91ab77ce2d9081f4c3ee928b4a1d22", "action": "login"},
    {"uid": 2, "email": "anna.nowak@example.com", "token": "b19f00a8e2c73499c0a1fa20433d7f12", "action": "sync"},
    {"uid": 3, "email": "piotr.zielinski@example.com", "token": "cc88d19a7e0065be44f1a3b90f0bb765", "action": "login"},
    {"uid": 4, "email": "ewa.wisniewska@example.com", "token": "1ae4f7c0107d8db922f09ab3f820112e", "action": "push"},
    {"uid": 5, "email": "ola.wojcik@example.com", "token": "9bb12f1140a9cde381f0ac55d1d9e008", "action": "sync"},
    {"uid": 6, "email": "marek.kaczmarek@example.com", "token": "0f661a2b7d9a300c1198fef7b102aa61", "action": "login"},
]
```

### Co wpisać do `final_answer`
`canonical_json({"leak_vuln": ..., "leak_v1": ..., "leak_v2": ..., "leak_reduction_pct": ..., "best_variant": "...", "sha256_logs_v2": "..."})`
`sha256_logs_v2 = sha256_text(canonical_json(logs_v2))`


In [ ]:
import matplotlib.pyplot as plt

LOG_EVENTS = [
    {"uid": 1, "email": "jan.kowalski@example.com", "token": "4f91ab77ce2d9081f4c3ee928b4a1d22", "action": "login"},
    {"uid": 2, "email": "anna.nowak@example.com", "token": "b19f00a8e2c73499c0a1fa20433d7f12", "action": "sync"},
    {"uid": 3, "email": "piotr.zielinski@example.com", "token": "cc88d19a7e0065be44f1a3b90f0bb765", "action": "login"},
    {"uid": 4, "email": "ewa.wisniewska@example.com", "token": "1ae4f7c0107d8db922f09ab3f820112e", "action": "push"},
    {"uid": 5, "email": "ola.wojcik@example.com", "token": "9bb12f1140a9cde381f0ac55d1d9e008", "action": "sync"},
    {"uid": 6, "email": "marek.kaczmarek@example.com", "token": "0f661a2b7d9a300c1198fef7b102aa61", "action": "login"},
]

# TODO: 3 warianty logowania + metryki wycieku + tabela + wykres
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM01", final_answer)


# BSM02 — Integralność: podmiana danych i porównanie CRC vs HMAC

### Co robisz
Symulujesz celową modyfikację komunikatów i porównujesz wykrywalność dla CRC oraz HMAC.

### Kroki
1. Użyj `PAYLOADS`, `TAMPER_IDS`, `SECRET_KEY`.
2. Format obliczeń (obowiązkowy):
   - CRC: `crc_hex = format(zlib.crc32(msg.encode("utf-8")) & 0xffffffff, "08x")`
   - HMAC: `hmac_hex = hmac.new(SECRET_KEY, msg.encode("utf-8"), hashlib.sha256).hexdigest()`
3. Wariant A (CRC):
   - liczysz CRC dla oryginału,
   - dla `TAMPER_IDS` zmieniasz `kwota=100` -> `kwota=900`,
   - atakujący przelicza CRC po zmianie,
   - serwer porównuje CRC i ocenia `crc_ok`.
4. Wariant B (HMAC):
   - liczysz HMAC dla oryginału,
   - wykonujesz tę samą modyfikację,
   - atakujący **nie** zna klucza, więc podpis nie jest aktualizowany,
   - serwer porównuje HMAC i ocenia `hmac_ok`.
5. Zbuduj tabelę rekordów z polami:
   `id, tampered, crc_original, crc_received, crc_ok, hmac_original, hmac_received, hmac_ok`.
6. Narysuj wykres liczby niewykrytych modyfikacji (`undetected_crc`, `undetected_hmac`).

### Sprawdź poprawność
- `tampered_total == 2`.
- `undetected_hmac <= undetected_crc`.

### Dane
```python
PAYLOADS = [
    {"id": "P01", "msg": "uid=1;kwota=100;waluta=PLN"},
    {"id": "P02", "msg": "uid=2;kwota=100;waluta=PLN"},
    {"id": "P03", "msg": "uid=3;kwota=100;waluta=PLN"},
    {"id": "P04", "msg": "uid=4;kwota=100;waluta=PLN"},
    {"id": "P05", "msg": "uid=5;kwota=100;waluta=PLN"},
    {"id": "P06", "msg": "uid=6;kwota=100;waluta=PLN"},
]
SECRET_KEY = b"BSM_L1_SECRET"
TAMPER_IDS = {"P02", "P05"}
```

### Co wpisać do `final_answer`
`canonical_json({"tampered_total": ..., "undetected_crc": ..., "undetected_hmac": ..., "better_method": "...", "sha256_table": "..."})`
`sha256_table = sha256_text(canonical_json(table_rows))`


In [ ]:
import zlib
import hmac
import hashlib
import json
import matplotlib.pyplot as plt

PAYLOADS = [
    {"id": "P01", "msg": "uid=1;kwota=100;waluta=PLN"},
    {"id": "P02", "msg": "uid=2;kwota=100;waluta=PLN"},
    {"id": "P03", "msg": "uid=3;kwota=100;waluta=PLN"},
    {"id": "P04", "msg": "uid=4;kwota=100;waluta=PLN"},
    {"id": "P05", "msg": "uid=5;kwota=100;waluta=PLN"},
    {"id": "P06", "msg": "uid=6;kwota=100;waluta=PLN"},
]
SECRET_KEY = b"BSM_L1_SECRET"
TAMPER_IDS = {"P02", "P05"}

# TODO 1: funkcje crc32_msg(msg) i hmac_msg(msg, key)
# TODO 2: przygotuj rekordy bazowe (id, msg, crc, hmac)
# TODO 3: zasymuluj podmianę kwoty dla TAMPER_IDS
# TODO 4: policz undetected_crc i undetected_hmac
# TODO 5: narysuj wykres słupkowy
# TODO 6: przygotuj final_answer jako JSON string


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM02", final_answer)


# BSM03 — Dostępność: awarie i strategia ponawiania żądań

### Co robisz
Mierzysz wpływ retry na dostępność usługi przy tej samej serii awarii.

### Kroki
1. Użyj `REQUESTS` i `MAX_RETRY`.
2. Policz trzy warianty:
   - `no_retry` (0 ponowień),
   - `retry_1` (1 ponowienie),
   - `retry_2` (2 ponowienia).
3. Dla każdego wariantu wylicz:
   - skuteczność [%] jako `100 * sukcesy / liczba_żądań`,
   - średnią liczbę prób,
   - liczbę żądań nieudanych.
4. Zbuduj tabelę porównawczą 3 wariantów.
5. Narysuj dwa wykresy:
   - skuteczność,
   - średnia liczba prób.

### Sprawdź poprawność
- `success_no_retry`, `success_retry_1`, `success_retry_2` zapisuj jako **procenty** (`float`) z dokładnością do 2 miejsc.
- Skuteczność nie może maleć przy większej liczbie retry.
- `retry_2` ma najwyższą dostępność.

### Dane
```python
REQUESTS = {
  "R01": [0, 1], "R02": [1], "R03": [0, 0, 1], "R04": [0, 0, 0],
  "R05": [1], "R06": [0, 1], "R07": [1], "R08": [0, 1],
  "R09": [0, 0, 1], "R10": [1], "R11": [0, 0, 0], "R12": [1]
}
MAX_RETRY = 2
```

### Co wpisać do `final_answer`
`canonical_json({"success_no_retry": ..., "success_retry_1": ..., "success_retry_2": ..., "best_policy": "..."})`
Wartości `success_*` podaj jako procenty (0-100) po `round(x, 2)`.


In [ ]:
import matplotlib.pyplot as plt

REQUESTS = {
  "R01": [0, 1], "R02": [1], "R03": [0, 0, 1], "R04": [0, 0, 0],
  "R05": [1], "R06": [0, 1], "R07": [1], "R08": [0, 1],
  "R09": [0, 0, 1], "R10": [1], "R11": [0, 0, 0], "R12": [1]
}
MAX_RETRY = 2

# TODO: obliczenia metryk i wykres
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM03", final_answer)


# BSM04 — Modelowanie: granice zaufania i ryzykowne przepływy

### Co robisz
Tworzysz prosty model przepływów i liczysz ryzyko na podstawie cech technicznych.

### Kroki
1. Użyj `FLOWS_TM` i `ZONE`.
2. Dla każdego przepływu policz cechy:
   - `cross_boundary`,
   - `insecure_channel`,
   - `no_auth`.
3. Zdefiniuj składowe ryzyka:
   - `r_boundary = 2*cross_boundary`,
   - `r_channel = 2*insecure_channel`,
   - `r_auth = 1*no_auth`,
   - `risk_total = r_boundary + r_channel + r_auth`.
4. Zbuduj pełną tabelę z tymi kolumnami.
5. Posortuj malejąco po `risk_total`, potem rosnąco po `id`.
6. Narysuj TOP 6 przepływów.

### Sprawdź poprawność
- Tabela ma dokładnie 8 wierszy.
- Każdy `risk_total` jest całkowity i z zakresu 0..5.

### Dane
```python
FLOWS_TM = [
  {"id":"F01","src":"mobile","dst":"api","ch":"https","auth":"jwt"},
  {"id":"F02","src":"mobile","dst":"analytics","ch":"http","auth":"none"},
  {"id":"F03","src":"api","dst":"db","ch":"tls","auth":"mtls"},
  {"id":"F04","src":"partner","dst":"api","ch":"https","auth":"api_key"},
  {"id":"F05","src":"mobile","dst":"backup","ch":"http","auth":"none"},
  {"id":"F06","src":"admin","dst":"api","ch":"https","auth":"password"},
  {"id":"F07","src":"api","dst":"cache","ch":"tls","auth":"service_token"},
  {"id":"F08","src":"mobile","dst":"api","ch":"https","auth":"none"},
]
ZONE = {
 "mobile":"untrusted", "partner":"untrusted", "admin":"untrusted",
 "api":"trusted", "db":"trusted", "cache":"trusted",
 "analytics":"external", "backup":"external"
}
```

### Co wpisać do `final_answer`
`canonical_json({"top3_flow_ids": [...], "cross_boundary_count": ..., "avg_risk_total": ...})`


In [ ]:
import matplotlib.pyplot as plt

FLOWS_TM = [
  {"id":"F01","src":"mobile","dst":"api","ch":"https","auth":"jwt"},
  {"id":"F02","src":"mobile","dst":"analytics","ch":"http","auth":"none"},
  {"id":"F03","src":"api","dst":"db","ch":"tls","auth":"mtls"},
  {"id":"F04","src":"partner","dst":"api","ch":"https","auth":"api_key"},
  {"id":"F05","src":"mobile","dst":"backup","ch":"http","auth":"none"},
  {"id":"F06","src":"admin","dst":"api","ch":"https","auth":"password"},
  {"id":"F07","src":"api","dst":"cache","ch":"tls","auth":"service_token"},
  {"id":"F08","src":"mobile","dst":"api","ch":"https","auth":"none"},
]
ZONE = {
 "mobile":"untrusted", "partner":"untrusted", "admin":"untrusted",
 "api":"trusted", "db":"trusted", "cache":"trusted",
 "analytics":"external", "backup":"external"
}

# TODO: obliczenia risk, sortowanie, wykres
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM04", final_answer)


# BSM05 — STRIDE: mapowanie zagrożeń dla przepływów

### Co robisz
Automatycznie przypisujesz klasy STRIDE do przepływów i liczysz ich rozkład.

### Kroki
1. Użyj tabeli z BSM04 (lub przelicz od zera).
2. Reguły mapowania:
   - `auth == "none"` -> `Spoofing`,
   - `ch == "http"` -> `Tampering`, `InformationDisclosure`,
   - `auth == "password"` -> `ElevationOfPrivilege`,
   - `cross_boundary == True` -> `Repudiation`,
   - `dst == "api" and auth == "none"` -> `DenialOfService`.
3. Dla każdego `id` utwórz posortowaną listę klas STRIDE.
4. Policz globalne liczniki klas.
5. Narysuj wykres słupkowy liczności klas.
6. Dodaj krótką tabelę: `klasa -> liczność -> procent`.

### Sprawdź poprawność
- W słowniku wynikowym musi pojawić się minimum 4 różne klasy STRIDE.
- Suma liczników = łączna liczba przypisań STRIDE.

### Co wpisać do `final_answer`
`canonical_json({"stride_counts": {...}, "most_common_stride": "...", "unique_stride_classes": ...})`


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Użyj FLOWS_TM i ZONE z BSM04

# TODO: mapowanie STRIDE, zliczenia, wykres
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM05", final_answer)


# BSM06 — Ryzyko: macierz 5x5 i heatmapa

### Co robisz
Wyznaczasz priorytet zagrożeń na podstawie `likelihood` i `impact`.

### Kroki
1. Wejście: `stride_counts` z BSM05.
2. Dla każdej klasy policz:
   - `likelihood = min(5, 1 + count)`,
   - `impact` z mapy:
     - `Spoofing=3`, `Tampering=4`, `Repudiation=2`,
     - `InformationDisclosure=5`, `DenialOfService=3`, `ElevationOfPrivilege=5`,
   - `score = likelihood * impact`.
3. Nadaj poziom:
   - `Low` (1-6), `Medium` (7-12), `High` (13-19), `Critical` (20-25).
4. Zbuduj tabelę: `class, likelihood, impact, score, level`.
5. Zbuduj macierz 5x5 i heatmapę (`imshow`) wg specyfikacji:
   - utwórz `M = np.zeros((5,5), dtype=int)`,
   - wiersz: `r = likelihood - 1` (0..4),
   - kolumna: `c = impact - 1` (0..4),
   - agregacja: `M[r, c] += 1` dla każdej klasy STRIDE,
   - oś Y rośnie od 1 do 5 z góry na dół (indeks wiersza 0 to likelihood=1),
   - oś X rośnie od 1 do 5 od lewej do prawej (indeks kolumny 0 to impact=1).

### Sprawdź poprawność
- Każdy `score` mieści się w 1..25.
- Liczba pozycji w tabeli = liczba klas z BSM05.
- `M.sum()` = liczba klas STRIDE użytych w tabeli ryzyka.

### Co wpisać do `final_answer`
`canonical_json({"top3_scores": [...], "critical_count": ..., "high_count": ..., "max_score_class": "..."})`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TODO: policz score per STRIDE
# TODO: zbuduj macierz 5x5 i heatmapę
# TODO: przygotuj final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM06", final_answer)


# BSM07 — Redukcja ryzyka: porównanie przed i po kontrolach

### Co robisz
Stosujesz kontrole bezpieczeństwa i liczysz ryzyko rezydualne.

### Kroki
1. Wejście: tabela `class, score` z BSM06.
2. Użyj współczynników redukcji:
```python
REDUCTION = {
  "Spoofing":0.35, "Tampering":0.30, "Repudiation":0.20,
  "InformationDisclosure":0.40, "DenialOfService":0.25, "ElevationOfPrivilege":0.45
}
```
3. Dla każdej klasy policz:
   - `residual = round(score * (1 - REDUCTION[class]), 2)`,
   - `delta = round(score - residual, 2)`.
4. Zbuduj tabelę `before/after/delta`.
5. Narysuj wykres grupowany `before vs after`.
6. Policz:
   - `avg_before`, `avg_after`,
   - `reduction_pct_total` dla sumy score.

### Sprawdź poprawność
- Dla każdej klasy `residual <= score`.
- `reduction_pct_total` > 0.

### Co wpisać do `final_answer`
`canonical_json({"avg_before": ..., "avg_after": ..., "reduction_pct_total": ..., "still_high": [...]})`
`still_high` = posortowana lista klas z `residual >= 13`.
Pusta lista `still_high` jest poprawna i może wystąpić dla tych danych.


In [ ]:
import matplotlib.pyplot as plt

REDUCTION = {
  "Spoofing":0.35, "Tampering":0.30, "Repudiation":0.20,
  "InformationDisclosure":0.40, "DenialOfService":0.25, "ElevationOfPrivilege":0.45
}

# TODO: oblicz residual i metryki
# TODO: wykres before/after
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM07", final_answer)


# BSM08 — Incydenty bezpieczeństwa: wpływ na A/I/C i priorytet

### Co robisz
Analizujesz incydenty i wyznaczasz priorytet na podstawie wpływu na A/I/C.

### Kroki
1. Użyj `INCIDENTS`.
2. Dla każdego incydentu policz:
   - `impact_score = 3*C + 2*I + 2*A`.
3. Priorytet (obowiązkowe progi):
   - `P1`, jeśli `impact_score >= 5`,
   - `P2`, jeśli `impact_score in {3, 4}`,
   - `P3`, jeśli `impact_score <= 2`.
4. Zbuduj tabelę: `id, type, A, I, C, impact_score, priority`.
5. Policz `p1`, `p2`, `p3`.
6. Narysuj:
   - wykres `P1/P2/P3`,
   - wykres średniego wpływu na A, I, C.

### Sprawdź poprawność (wartości referencyjne)
- `p1 == 4`, `p2 == 5`, `p3 == 3`.
- `top_incident_id == "I04"` przy sortowaniu malejąco po `impact_score`, a przy remisie rosnąco po `id`.
- `top_impact_score == 5`.

### Dane
```python
INCIDENTS = [
 {"id":"I01","type":"token_leak","A":0,"I":0,"C":1},
 {"id":"I02","type":"tamper_payment","A":0,"I":1,"C":0},
 {"id":"I03","type":"api_timeout","A":1,"I":0,"C":0},
 {"id":"I04","type":"credential_stuffing","A":0,"I":1,"C":1},
 {"id":"I05","type":"log_exposure","A":0,"I":0,"C":1},
 {"id":"I06","type":"replay_attack","A":0,"I":1,"C":1},
 {"id":"I07","type":"db_lock","A":1,"I":1,"C":0},
 {"id":"I08","type":"backup_leak","A":0,"I":0,"C":1},
 {"id":"I09","type":"config_error","A":1,"I":1,"C":0},
 {"id":"I10","type":"session_hijack","A":0,"I":1,"C":1},
 {"id":"I11","type":"service_crash","A":1,"I":0,"C":0},
 {"id":"I12","type":"exported_component","A":0,"I":1,"C":1},
]
```

### Co wpisać do `final_answer`
`canonical_json({"p1": ..., "p2": ..., "p3": ..., "top_incident_id": "...", "top_impact_score": ...})`


In [ ]:
import matplotlib.pyplot as plt

INCIDENTS = [
 {"id":"I01","type":"token_leak","A":0,"I":0,"C":1},
 {"id":"I02","type":"tamper_payment","A":0,"I":1,"C":0},
 {"id":"I03","type":"api_timeout","A":1,"I":0,"C":0},
 {"id":"I04","type":"credential_stuffing","A":0,"I":1,"C":1},
 {"id":"I05","type":"log_exposure","A":0,"I":0,"C":1},
 {"id":"I06","type":"replay_attack","A":0,"I":1,"C":1},
 {"id":"I07","type":"db_lock","A":1,"I":1,"C":0},
 {"id":"I08","type":"backup_leak","A":0,"I":0,"C":1},
 {"id":"I09","type":"config_error","A":1,"I":1,"C":0},
 {"id":"I10","type":"session_hijack","A":0,"I":1,"C":1},
 {"id":"I11","type":"service_crash","A":1,"I":0,"C":0},
 {"id":"I12","type":"exported_component","A":0,"I":1,"C":1},
]

# TODO: impact_score, priority, tabela, 2 wykresy
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM08", final_answer)


# BSM09 — Mini Threat Model v1: raport JSON + radar AIC

### Co robisz
Składasz jeden artefakt modelowania bezpieczeństwa na podstawie wyników BSM01-BSM08.

### Kroki
1. Wczytaj poprzednie wyniki:
```python
b1 = json.loads(answers["BSM01"])
b2 = json.loads(answers["BSM02"])
b3 = json.loads(answers["BSM03"])
b4 = json.loads(answers["BSM04"])
b5 = json.loads(answers["BSM05"])
b6 = json.loads(answers["BSM06"])
b7 = json.loads(answers["BSM07"])
b8 = json.loads(answers["BSM08"])
```
2. Zdefiniuj metryki pośrednie (obowiązkowe wzory):
   - `leak_reduction_pct = round(100 * (b1["leak_vuln"] - b1["leak_v2"]) / b1["leak_vuln"], 2)`
   - `improvement_pct = round(b3["success_retry_2"] - b3["success_no_retry"], 2)`  # różnica w punktach procentowych
   - `avg_after = b7["avg_after"]`
3. Wyznacz `aic_score` (0-100):
   - `C = min(100, max(0, leak_reduction_pct))`
   - `I = min(100, max(0, 100 - 5*avg_after))`
   - `A = min(100, max(0, improvement_pct))`  # wejście: wartość procentowa z BSM03
4. Zbuduj `model_v1` z sekcjami:
   `confidentiality, integrity, availability, trust_boundaries, stride, risk, residual, incidents, aic_score, generated_at`.
5. Ustaw `generated_at = "BSM_L1"`.
6. Narysuj radar A-I-C.
7. Zserializuj model: `model_json = canonical_json(model_v1)`.

### Sprawdź poprawność (wartości referencyjne)
- `len(model_json) > 200`.
- `aic_min <= aic_max`.
- `generated_at == "BSM_L1"`.

### Co wpisać do `final_answer`
`canonical_json({"json_len": ..., "sha256_model_v1": "...", "aic_min": ..., "aic_max": ...})`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TODO: zbuduj model_v1 na podstawie wyników BSM01-BSM08
# TODO: policz aic_score i narysuj radar A-I-C
# TODO: model_json = canonical_json(model_v1)
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM09", final_answer)


# BSM10 — Podpis końcowy lekcji

### Co robisz
Tworzysz reprodukowalny podpis całego zestawu wyników.

### Kroki
1. Użyj słownika `answers`, który był zapisywany po każdym zadaniu.
2. Zbuduj listę:
```python
answers_ordered = [
  answers["BSM01"], answers["BSM02"], answers["BSM03"], answers["BSM04"], answers["BSM05"],
  answers["BSM06"], answers["BSM07"], answers["BSM08"], answers["BSM09"]
]
```
3. Połącz: `joined = "||".join(answers_ordered)`.
4. Policz:
   - `course_hash = sha256_text(joined)`,
   - `joined_len = len(joined)`,
   - `unique_chars = len(set(joined))`.
5. Policz `top3_chars` jako 3 najczęstsze znaki z `joined`.
6. Narysuj histogram częstości znaków.

### Sprawdź poprawność
- `course_hash` ma długość 64.
- `len(answers_ordered) == 9`.
- `joined_len > 100`.

### Co wpisać do `final_answer`
`canonical_json({"course_hash": "...", "joined_len": ..., "unique_chars": ..., "top3_chars": [...]})`


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# TODO: uzupełnij final_answer z BSM01-BSM09 w słowniku answers (przez zapisz_i_wyslij)

# TODO: answers_ordered = [answers["BSM01"], ..., answers["BSM09"]]
# TODO: joined = "||".join(answers_ordered)
# TODO: oblicz course_hash, joined_len, unique_chars
# TODO: top3_chars = 3 najczęstsze znaki
# TODO: histogram znaków
# final_answer = canonical_json({...})


In [ ]:
final_answer = ""
zapisz_i_wyslij("BSM10", final_answer)
